# Bias and fairness - part 1: audit the model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/bias-fairness/notebooks/ex4_audit.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Bias and fairness*

The Heart Disease XGBoost of the explainability lab - the same split, the same
model - asked a new question: does it treat women and men the same way?

**Four holes.** The fairness metrics by hand, one number per definition,
the same audit over twenty random splits, and the model without `sex`.

# Data and model

UCI Heart Disease, Cleveland (CC BY 4.0), stored in the course repository.
`sex` is 1 for men and 0 for women.

In [ ]:
%pip install -q fairlearn

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_cleveland.csv"
CSV = Path("heart_disease_cleveland.csv")
if not CSV.is_file():
    urllib.request.urlretrieve(URL, CSV)

SEED = 42
N_SPLITS = 20
N_DRAWS = 200
GROUPS = {0: "female", 1: "male"}


def load_heart(path: Path = CSV) -> tuple[pd.DataFrame, pd.Series]:
    """13 features (missing values filled with the median) and disease = num > 0."""
    table = pd.read_csv(path)
    y = (table.pop("num") > 0).astype(int)
    return table.fillna(table.median(numeric_only=True)), y


class XGBClassifier64(xgb.XGBClassifier):
    """XGBoost whose predict_proba returns float64: `fairlearn`'s
    ThresholdOptimizer writes its interpolated probabilities into an array of
    the model's output dtype and fails on XGBoost's float32. The numbers are
    the same."""

    def predict_proba(self, X, *args, **kwargs):
        return super().predict_proba(X, *args, **kwargs).astype(np.float64)


def train_xgboost(X_train: pd.DataFrame, y_train: pd.Series, seed: int = SEED):
    """The shared recipe of the tabular running example."""
    model = XGBClassifier64(n_estimators=200, max_depth=3, learning_rate=0.1,
                            eval_metric="logloss", random_state=seed)
    return model.fit(X_train, y_train)


def wilson(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """The Wilson score interval of a proportion (95 % by default)."""
    p = successes / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


X, y = load_heart()
for code, name in GROUPS.items():
    print(f"{name}: {(X['sex'] == code).sum()} patients, {y[X['sex'] == code].mean():.1%} with the disease")

In [ ]:
# The model of the explainability lab: the 75 / 25 split, seed 42.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    random_state=SEED, stratify=y)
model = train_xgboost(X_train, y_train)
y_pred = model.predict(X_test)
print(f"test accuracy {(y_pred == y_test.to_numpy()).mean():.3f} on {len(y_test)} patients")

# The metrics, by group

### Hole 1 - selection rate, TPR, FPR, PPV

In [ ]:
# TODO 1: the fairness metrics by hand. For each group (the `sex` column:
#         0 = female, 1 = male) return a row with
#           n               the number of patients,
#           sick            the number with the disease,
#           base rate       the share with the disease,
#           selection rate  the share the model flags (y_pred == 1),
#           TPR             the share of the SICK patients it flags,
#           FPR             the share of the HEALTHY patients it flags,
#           PPV             the share of the FLAGGED patients who are sick.
#         One row per group name of GROUPS, as a DataFrame.

def group_metrics(y_true, y_pred, sex):
    y_true, y_pred, sex = np.asarray(y_true), np.asarray(y_pred), np.asarray(sex)
    rows = {}
    for code, name in GROUPS.items():
        ...
    return pd.DataFrame(rows).T


metrics = group_metrics(y_test, y_pred, X_test["sex"])
metrics.round(3)

In [ ]:
metrics[["selection rate", "TPR", "FPR", "PPV"]].T.plot.bar(
    figsize=(7, 4), rot=0, color=["#2a78d6", "#eb6834"])
plt.ylabel("rate"); plt.title("the explainability split by sex, 76 test patients")
plt.show()

### Hole 2 - one number per definition

Which definition says this model is unfair - and by how much?

In [ ]:
# TODO 2: one number per definition, male minus female:
#         the demographic-parity gap = |difference of the selection rates|,
#         the equalized-odds gap     = the larger of |TPR difference| and |FPR difference|.
#         Return them in a dict together with the four signed differences
#         ("selection rate", "TPR", "FPR", "PPV").

def gaps(metrics):
    ...


gaps(metrics)

### How sure are we?

A rate of a small group is a count divided by a small number. The cell below
prints the counts and a 95 % (Wilson) interval for each TPR and FPR.

In [ ]:
# The counts behind the rates, with a 95 % Wilson interval (`wilson` above).
sex, sick = X_test["sex"].to_numpy(), y_test.to_numpy()
for code, name in GROUPS.items():
    t, p = sick[sex == code], y_pred[sex == code]
    for metric, k, n in (("TPR", p[t == 1].sum(), (t == 1).sum()),
                         ("FPR", p[t == 0].sum(), (t == 0).sum())):
        lo, hi = wilson(k, n)
        print(f"{name:6s} {metric} {k:2d}/{n:2d} = {k / n:.3f}   95 % interval [{lo:.2f}, {hi:.2f}]")

# The model, or the split?

There are only 26 women in the test part, 9 of them sick: one prediction
moves their TPR by 11 points.

### Hole 3 - twenty splits

In [ ]:
# TODO 3: is it the model, or the split? Repeat the whole audit - split
#         (test_size=0.25, stratify=y, random_state=seed), train (with that
#         seed), predict, group_metrics - for seed = 0 ... N_SPLITS - 1.
#         Collect one row per split and group in a DataFrame `repeated`
#         (columns: seed, group, and the metrics), and print for each metric
#         and group the median and the 10th and 90th percentile - and in how
#         many of the twenty splits the men's value is higher.

rows = []
for seed in range(N_SPLITS):
    ...
repeated = pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True)
for ax, metric in zip(axes, ["selection rate", "TPR", "FPR", "PPV"]):
    for k, (group, color) in enumerate([("female", "#2a78d6"), ("male", "#eb6834")]):
        v = repeated.loc[repeated["group"] == group, metric]
        ax.scatter(np.full(len(v), k) + np.random.uniform(-0.1, 0.1, len(v)), v, color=color, alpha=0.5)
        ax.plot([k - 0.25, k + 0.25], [metrics.loc[group, metric]] * 2, color="k", lw=2)
    ax.set_xticks([0, 1], ["women", "men"]); ax.set_title(metric)
axes[0].set_ylabel("rate (line: the explainability split, dots: 20 splits)")
plt.show()

# Without `sex`

The simplest fix that comes to mind: do not show the model the sex at all.

### Hole 4 - the twenty splits without `sex`

In [ ]:
# TODO 4: "fairness through unawareness". Repeat the twenty splits of hole 3
#         with a model trained WITHOUT the `sex` column - the audit still
#         slices the test part by X_test["sex"]. Collect `unaware` like
#         `repeated`, and print for the selection rate, TPR and FPR the median
#         over the splits of (men minus women), with sex and without it.

features = [c for c in X.columns if c != "sex"]
rows = []
for seed in range(N_SPLITS):
    ...
unaware = pd.DataFrame(rows)

In [ ]:
# Where does sex hide? The same recipe predicts sex from the other twelve
# features, on twenty splits.
auc = []
for seed in range(N_SPLITS):
    Ztr, Zte, s_tr, s_te = train_test_split(X[features], X["sex"], test_size=0.25,
                                            random_state=seed, stratify=X["sex"])
    auc.append(roc_auc_score(s_te, train_xgboost(Ztr, s_tr, seed).predict_proba(Zte)[:, 1]))
print(f"sex from the other twelve features: AUC median {np.median(auc):.2f} "
      f"(range {min(auc):.2f}-{max(auc):.2f})")
for code, name in GROUPS.items():
    print(f"{name}: a normal thallium scan (thal = 3) in {(X.loc[X['sex'] == code, 'thal'] == 3).mean():.0%}")

# Questions

1. Which definition finds a gap on the explainability split, and which does
   not? Compare the selection-rate gap with the difference in prevalence.
2. Look at the intervals of that split. Could you claim from it alone that
   the TPR or the FPR differs between women and men?
3. After twenty splits: which gaps are stable, and which one did the
   explainability split hide?
4. Without `sex` the gaps shrank. Why did they not disappear - and would you
   remove `sex` from a model for a cardiac work-up?
5. The model flags patients for a cardiac work-up. Which definition would you
   hold it to - and why is that a clinical decision, not a statistical one?

---

*The numbers are computed by [`heart.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/bias-fairness/code/heart.py) of
the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/bias-fairness/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*